# Model Building & Comparison

In this notebook, we will use the 110 engineered features developed in the previous phase and compare different machine learning models for the four electricity demand targets.

The goal is to identify which model architecture can outperform our current XGBoost benchmark while keeping the validation setup identical across all experiments. We will first compare models fairly and only tune the strongest candidates afterward.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

DATA_DIR = Path("../data")

train = pd.read_csv(DATA_DIR / "train.csv")
validation = pd.read_csv(DATA_DIR / "validation.csv")
test = pd.read_csv(DATA_DIR / "test_input.csv")

for df in [train, validation, test]:
    df["datetime"] = pd.to_datetime(
        df["datetime"],
        format="%d-%m-%Y %H:%M"
    )

targets = [
    "nat_demand",
    "load_tocumen_mwh",
    "load_santiago_mwh",
    "load_david_mwh"
]

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (34343, 51)
Validation: (7248, 51)
Test: (2352, 47)


In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.features import build_features, get_model_features, TARGETS

In [3]:
train_fe = build_features(train)
validation_fe = build_features(validation)
test_fe = build_features(test)

model_features = get_model_features(train_fe)

print("Train:", train_fe.shape)
print("Validation:", validation_fe.shape)
print("Test:", test_fe.shape)
print("Model features:", len(model_features))

Train: (34343, 118)
Validation: (7248, 118)
Test: (2352, 114)
Model features: 110


In [4]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

window_counts = validation_fe.groupby("window_id").size()
complete_windows = window_counts[window_counts == 168].index

val_complete = validation_fe[
    validation_fe["window_id"].isin(complete_windows)
].copy()

print("Complete validation windows:", len(complete_windows))
print("Validation rows:", len(val_complete))


def evaluate_predictions(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE (%)": np.mean(
            np.abs((y_true - y_pred) / y_true)
        ) * 100,
        "R²": r2_score(y_true, y_pred)
    }

Complete validation windows: 34
Validation rows: 5712


In [5]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

models = {
    "XGBoost": XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ),

    "LightGBM": LGBMRegressor(
        n_estimators=500,
        max_depth=-1,
        num_leaves=31,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="regression",
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ),

    "CatBoost": CatBoostRegressor(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1
    )
}

comparison_results = []
model_predictions = {}

X_train = train_fe[model_features]
X_val = val_complete[model_features]

for model_name, model in models.items():
    print(f"\n===== {model_name} =====")

    for target in targets:
        print(f"Training {target}...")

        model.fit(X_train, train_fe[target])

        y_true = val_complete[target]
        y_pred = model.predict(X_val)

        model_predictions[(model_name, target)] = y_pred

        metrics = evaluate_predictions(y_true, y_pred)

        comparison_results.append({
            "Model": model_name,
            "Target": target,
            **metrics
        })

comparison_results = pd.DataFrame(comparison_results)

display(comparison_results.round(4))


===== XGBoost =====
Training nat_demand...
Training load_tocumen_mwh...
Training load_santiago_mwh...
Training load_david_mwh...

===== LightGBM =====
Training nat_demand...
Training load_tocumen_mwh...
Training load_santiago_mwh...
Training load_david_mwh...

===== CatBoost =====
Training nat_demand...
Training load_tocumen_mwh...
Training load_santiago_mwh...
Training load_david_mwh...


,Model,Target,MAE,RMSE,MAPE (%),R²
0,XGBoost,nat_demand,45.4564,63.8399,4.0316,0.8866
1,XGBoost,load_tocumen_mwh,37.0656,52.0682,4.0220,0.8875
2,XGBoost,load_santiago_mwh,2.9409,4.1086,4.0612,0.8869
3,XGBoost,load_david_mwh,5.3644,7.5137,4.0815,0.8859
4,LightGBM,nat_demand,45.4881,63.9313,4.0404,0.8863
5,LightGBM,load_tocumen_mwh,37.2435,52.4061,4.0353,0.8860
6,LightGBM,load_santiago_mwh,2.9578,4.1426,4.0908,0.8850
7,LightGBM,load_david_mwh,5.3555,7.5438,4.0755,0.8850
8,CatBoost,nat_demand,44.4172,62.9249,3.9524,0.8898
9,CatBoost,load_tocumen_mwh,35.9472,51.2374,3.9120,0.8910


In [6]:
comparison_results.to_csv(
    "../outputs/metrics/model_comparison.csv",
    index=False
)

## Conclusion

In this notebook, we compared XGBoost, LightGBM and CatBoost using the same 110 engineered features, the same 34 complete validation windows, and the same evaluation metrics.

CatBoost performed best across all four forecasting targets, reaching MAPE values of about 3.91–3.99% and R² values around 0.89. This is an improvement over our earlier XGBoost benchmark.

Based on these results, CatBoost is our current candidate for further optimization. The next step is to tune its hyperparameters and check whether the improvement remains consistent across additional time based validation windows.